### Latent Factor Vasicek Model
Considering a homogenous risk class, every oligor $i$ in the class has the following asset return at time $t$:
$$Z_{i,t} = \sqrt{\rho} \cdot X_t + \sqrt{1-\rho} \cdot \varepsilon_{i,t}$$

where:
- $X_t \sim N(0,1)$ is the systematic (market/economy) factor - this impacts all obligors
- $\varepsilon_{i,t} \sim N(0,1)$ is the idiosyncratic risk (specific to obligor $i$)
- $\rho \in [0,1]$ is the factor loading (squared because we assume N(0,1)) - asset correlation to the factor, **(assumed constant here, obviously a problem)**
- Obligor $i$ defaults when $Z_{i,t} < \Phi^{-1}(PD_i)$ (standardized return of the asset falls below a specific threshold of uncond PD)
> I mentioned the temporal component here to highlight the constant $\rho$ assumption. 

### Conditional PD
Given the realization of the macro factor $X = x$:
$$PD_i(x) = \Phi\!\left(\frac{\Phi^{-1}(PD_i) - \sqrt{\rho_i}\, x}{\sqrt{1 - \rho_i}}\right)$$


> #### The idea in this model is that as the portfolio becomes large enough (such that no single exposure has a dominating proportion) the portfolio loss, *conditional on the factor*, becomes deterministic.

### Gaussian Copula - I will start with this for now
The Gaussian copula couples marginal $PD$ distributions using the multivariate normal:
$$C(u_1, \ldots, u_n) = \Phi_n\!\left(\Phi^{-1}(u_1), \ldots, \Phi^{-1}(u_n); R\right)$$

where $R$ is the correlation matrix, $\Phi_n$ is the $n$-dimensional N(0,1) cdf and $u_i = F_i(z_i)$ are the marginal CDFs.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats, optimize
from scipy.special import ndtri  # (inverse N(0,1) CDF)

np.random.seed(123)

---
## 1. Building the Default Rate 

The default rate of a cluster $k$ can be defined as:
$$\mu_k = \frac{\text{Number of defaults in cluster } k}{\text{Number of loans in cluster } k}$$

> $\mu_k$ is backwards-looking, but it can be used to estimate the (forward-looking) $PD_k$. 

In [170]:
df = pd.read_parquet('lending_club_subset.parquet')
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,home_ownership,annual_inc,issue_d,loan_status,purpose,addr_state,dti,fico_range_low,fico_range_high
0,3600.0,36 months,13.99,C,C4,MORTGAGE,55000.0,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,675.0,679.0
1,24700.0,36 months,11.99,C,C1,MORTGAGE,65000.0,Dec-2015,Fully Paid,small_business,SD,16.06,715.0,719.0
2,20000.0,60 months,10.78,B,B4,MORTGAGE,63000.0,Dec-2015,Fully Paid,home_improvement,IL,10.78,695.0,699.0
3,35000.0,60 months,14.85,C,C5,MORTGAGE,110000.0,Dec-2015,Current,debt_consolidation,NJ,17.06,785.0,789.0
4,10400.0,60 months,22.45,F,F1,MORTGAGE,104433.0,Dec-2015,Fully Paid,major_purchase,PA,25.37,695.0,699.0


In [171]:
status_counts = df['loan_status'].value_counts() #distribution of loan statuses
for status, count in status_counts.items():
    print(f" {status:<55s} {count:>8,d} ({count/len(df):4.3%})")

 Fully Paid                                              1,076,751 (47.629%)
 Current                                                  878,317 (38.852%)
 Charged Off                                              268,559 (11.879%)
 Late (31-120 days)                                        21,467 (0.950%)
 In Grace Period                                            8,436 (0.373%)
 Late (16-30 days)                                          4,349 (0.192%)
 Does not meet the credit policy. Status:Fully Paid         1,988 (0.088%)
 Does not meet the credit policy. Status:Charged Off          761 (0.034%)
 Default                                                       40 (0.002%)


In [172]:
#  default rate = defaulted/total loans 

# Default can have different definitions, but 
# I exclude grace period & short-term late from defaulted statuses because they aren't necessarily defaults (yet, at least)
defaulted_statuses = {
    'Charged Off',
    'Default',
    'Late (31-120 days)',
    'Does not meet the credit policy. Status:Charged Off'
}

alive_statuses = {
    'Fully Paid',
    'Does not meet the credit policy. Status:Fully Paid',
    'Current'
}


default_or_alive_mask = df['loan_status'].isin(defaulted_statuses | alive_statuses)
df_filtered=df.loc[default_or_alive_mask]

df_filtered['is_default'] = df_filtered['loan_status'].isin(defaulted_statuses).astype(int)
# 1=default, 0=alive

default_count = df_filtered['is_default'].sum()
total_count = len(df_filtered)
default_rate = default_count / total_count

print(f"Defaults:    {default_count:,}")
print(f"Total loans: {total_count:,}")
print(f"Default rate (defaulted / total): {default_rate:.2%}")

Defaults:    290,827
Total loans: 2,247,883
Default rate (defaulted / total): 12.94%


/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_1556/102679427.py:22: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



### Default rates by grade. I also include average loan amount and average FICO score by grade since I'll use these stats to help interpret the copula results down the line.

In [173]:
grade_stats = (df_filtered.groupby('grade').agg(
    n_total=('is_default', 'size'),
    n_defaults=('is_default', 'sum'),
    avg_loan=('loan_amnt', 'mean'),
    avg_fico=('fico_range_low', 'mean'),
).sort_index())

grade_stats['pd_empirical'] = grade_stats['n_defaults'] / grade_stats['n_total'] #  default rate = defaulted/total loans 

print("Empirical Default Rates by Grade")
print(f"{'Grade':<8s} {'# Loans':>10s} {'# Defaults':>12s} {'Default Rate':>16s} {'Avg Loan':>10s} {'Avg FICO':>10s}")
for grade, row in grade_stats.iterrows():
    print(f"  {grade:<6s} {row['n_total']:>10,.0f} {row['n_defaults']:>12,.0f} "
          f"{row['pd_empirical']:>16.2%} {row['avg_loan']:>9,.0f} {row['avg_fico']:>10.0f}")
print(f"  {'Total':<6s} {grade_stats['n_total'].sum():>10,.0f} "
      f"{grade_stats['n_defaults'].sum():>12,.0f} "
      f"{(grade_stats['n_defaults'].sum() / grade_stats['n_total'].sum()):>16.2%}")

Empirical Default Rates by Grade
Grade       # Loans   # Defaults     Default Rate   Avg Loan   Avg FICO
  A         432,054       15,536            3.60%    14,599        729
  B         660,793       57,449            8.69%    14,164        700
  C         645,584       93,359           14.46%    15,021        689
  D         321,556       66,025           20.53%    15,693        684
  E         134,438       38,365           28.54%    17,441        682
  F          41,431       15,225           36.75%    19,106        680
  G          12,027        4,868           40.48%    20,361        679
  Total   2,247,883      290,827           12.94%


In [174]:
# default rate visualization
colors = ['#2ecc71', '#27ae60', '#f1c40f', '#e67e22', '#e74c3c', '#c0392b', '#8e44ad']
total_loans = grade_stats['n_total']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Default Rate by Grade", "Number of Loans by Grade"),
    horizontal_spacing=0.12
)

# Left panel: default rate
fig.add_trace(
    go.Bar(
        x=grade_stats.index,
        y=grade_stats['pd_empirical'],
        marker_color=colors[:len(grade_stats)],
        text=[f"{pd:.1%}" for pd in grade_stats['pd_empirical']],
        textposition='outside',
        name='Default Rate'
    ), row=1, col=1
)

# Right panel: total loan counts
fig.add_trace(
    go.Bar(
        x=grade_stats.index,
        y=total_loans,
        marker_color=colors[:len(grade_stats)],
        text=[f"{n:,.0f}" for n in total_loans],
        textposition='outside',
        name='# Loans'
    ), row=1, col=2
)

fig.update_yaxes(title_text="Default Rate", tickformat=".0%", row=1, col=1)
fig.update_yaxes(title_text="Number of Loans", row=1, col=2)
fig.update_layout(
    template="plotly_white", showlegend=False, height=450
)
fig.show()

- **Grade A** has the lowest default rate, **Grade G** the highest, as one would expect
- The left tail (clusters F, G) have fewer observations

Each grade defines a natural **cluster** of obligors with similar creditworthiness. I will use these clusters in building the copula.

---
## 2. Copula & Clustering Obligors

(We will not consider the temporal component here)

In a large portfolio (say,$n = 1{,}000{,}000+$ loans), modeling every pairwise default correlation is computationally unfeasible, so we make some simplifying assumptions and group obligors into **$K$ clusters** (i.e, the 7 Lending Club grades A–G).

We assume that:

1. **Within a cluster**, all obligors share the same marginal default probability $PD_k$ and factor loading $\sqrt{\rho_k}$
2. **Across clusters**, dependence is driven by the **common systematic factor** $X$

### The Gaussian Copula for Clustered Obligors

Let $D_k$ be the number of defaults in cluster $k$ (with $n_k$ obligors). Using the **Gaussian copula** for the joint default distribution works as follows:

**Step 1: Vasicek model for each obligor**

For obligor $i$ in cluster $k$:
$$Z_{k,i} = \sqrt{\rho_k}\, X + \sqrt{1 - \rho_k}\, \varepsilon_{k,i}$$

where $X, \varepsilon_{k,i} \stackrel{\text{iid}}{\sim} N(0,1)$. 
> **($ X,\varepsilon \sim N$  is a pretty strong assumption)**

**Step 2: Default threshold**

Obligor $i$ in cluster $k$ defaults if:
$$Z_{k,i} < \Phi^{-1}(PD_k) \equiv c_k$$

**Step 3: Conditional independence**

Given $X = x$, defaults within each cluster are **independent Bernoulli** trials (summing up to a **Binomial**):
$$D_k \mid X = x \sim \text{Binomial}\!\left(n_k,\; \Phi\!\left(\frac{c_k - \sqrt{\rho_k}\, x}{\sqrt{1-\rho_k}}\right)\right)$$

They are independent trials because we assume that dependence is based solely on $X$.

**Step 4: Copula and clusters**

The joint distribution of default counts across all $K$ clusters is obtained by integrating out $X$ (integrating over the distribution of the factor):
$$P(D_1 \leq d_1, \ldots, D_K \leq d_K) = \int_{-\infty}^{\infty} \prod_{k=1}^{K} P(D_k \leq d_k \mid X = x)\;\phi(x)\,dx$$

where $\phi$ is the standard normal density. We integrate over the distribution of $X$ in order to "average" over all possible values of it, weighted by their probabilit.

### Short form

The conditional $PD$ for cluster $k$, given $X=x$:
$$p_k(x) = \Phi\!\left(\frac{\Phi^{-1}(PD_k) - \sqrt{\rho_k}\, x}{\sqrt{1-\rho_k}}\right)$$

Then the **joint CDF of default counts** is:
$$F(d_1, \ldots, d_K) = \int_{-\infty}^{\infty} \prod_{k=1}^{K} \underbrace{B(d_k;\, n_k,\, p_k(x))}_{\text{Binomial CDF}} \;\phi(x)\,dx$$

In [175]:
grades = grade_stats.index.tolist()       # ['A', 'B', 'C', 'D', 'E', 'F', 'G']
K = len(grades)                           
pds = grade_stats['pd_empirical'].values  # empirical PD (default rates)
n_k = total_loans.values       # cluster size
thresholds = ndtri(pds)                   # c_k = phi⁻¹(PD_k), percentile func of N(0,1)

print(f"{'Grade':<8s} {'n_k':>10s} {'PD_k':>10s} {'c_k = phi⁻¹(PD)':>16s}")
for i, g in enumerate(grades):
    print(f"  {g:<6s} {n_k[i]:>10,d} {pds[i]:>10.4f} {thresholds[i]:>16.4f}")

Grade           n_k       PD_k  c_k = phi⁻¹(PD)
  A         432,054     0.0360          -1.7996
  B         660,793     0.0869          -1.3598
  C         645,584     0.1446          -1.0598
  D         321,556     0.2053          -0.8227
  E         134,438     0.2854          -0.5670
  F          41,431     0.3675          -0.3385
  G          12,027     0.4048          -0.2411


The thresholds $c_k$ are increasingly less negative as grade worsens.

So an economy shock doesn't have to be too extreme to trigger default for grades with poor creditworthiness.

---
## 3. Solving for Factor Loadings


While we do have the marginal default rates $PD_k$ from the data, we don't directly observe the factor loadings $\sqrt{\rho_k}$. The loadings control how strongly each cluster's $PD$ responds to the systematic factor.

### Moment matching by pairwise default correlation
If we can estimate the pairwise default correlation $\hat{\rho}^{\text{def}}_{k}$ within each cluster using historical data, then we can invert the relationship to find $\rho_k$.

The relationship between the asset correlation $\rho$ and default correlation for a homogeneous group:

$$\text{Corr}(D_i, D_j) = \frac{\Phi_2(\Phi^{-1}(PD), \Phi^{-1}(PD); \rho) - PD^2}{PD(1-PD)}$$

where $\Phi_2(\cdot, \cdot; \rho)$ is the bivariate normal CDF with correlation $\rho$.


We can also use the variance of the conditional default rate (observable from time-series data):

$$\text{Var}[p_k(X)] = E[p_k(X)^2] - PD_k^2$$

And the asset correlation satisfies the following:

$$\rho_k^{\text{def}} = \frac{\text{Var}[p_k(X)]}{PD_k(1 - PD_k)}$$

> It is worth noting that I only used moment matching here to get a quick approx, I will try a couple of other ways.

In [176]:
# Time-series moment matching 
# I parse the issue date and compute default rates by grade and quarter
df_filtered['issue_d'] = pd.to_datetime(df_filtered['issue_d'], format='%b-%Y')
df_filtered['quarter'] = df_filtered['issue_d'].dt.to_period('Q')

# Default rate by grade & quarter
ts = (
    df_filtered.groupby(['grade', 'quarter'])
    .agg(n=('is_default', 'size'), defaults=('is_default', 'sum'))
    .reset_index()
)
ts['pd_quarterly'] = ts['defaults'] / ts['n']

# keep the quarters that have enough data (at least 50 loans per grade)
ts = ts[ts['n'] >= 50]

print(f"{ts['quarter'].nunique()} quarters × {ts['grade'].nunique()} grades")
print(f"Quarter range: {ts['quarter'].min()} to {ts['quarter'].max()}")

45 quarters × 7 grades
Quarter range: 2007Q4 to 2018Q4


/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_1556/309982654.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/5r/3zb78l7x2cb8py_g4z_s_c040000gn/T/ipykernel_1556/309982654.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [177]:
# Now we estimate asset correlation from Var(conditional PD)
# For each grade, the variance of the quarterly default rate gives us info about how much the factor X moves the default rate.

# Under the single-factor model:
#   Var[rho_k(X)] ≈ rho_default * PD_k * (1 - PD_k)

# So: rho_default ≈ Var[rho_k(X)] / (PD_k * (1 - PD_k))
# lastly we invert rho_default → rho_asset using the bivariate normal.

from scipy.stats import norm

def invert_default_to_asset_correlation(pd_val, rho_default, tol=1e-8): #we solve for the asset correlation  via bisection
    from scipy.stats import multivariate_normal
    c = ndtri(pd_val)
    target = rho_default * pd_val * (1 - pd_val) + pd_val**2  

    def objective(rho_asset):
        if rho_asset <= 0:
            return ndtri(pd_val)**2 - target         
        # At ρ=0, BVN CDF = PD², which is always < target → returns negative
        # This guides brentq to search in the positive ρ direction
        # Bivariate N CDF
        cov = np.array([[1, rho_asset], [rho_asset, 1]])
        bvn_cdf = multivariate_normal.cdf([c, c], mean=[0, 0], cov=cov)
        return bvn_cdf - target

    # we do bisection on rho_asset
    try:
        rho_asset = optimize.brentq(objective, 1e-6, 0.999, xtol=tol)
    except ValueError:
        rho_asset = np.nan
    return rho_asset

rho_ts = {}
for i, g in enumerate(grades):
    ts_g = ts[ts['grade'] == g]['pd_quarterly']
    var_pd = ts_g.var()
    pd_val = pds[i]
    rho_def = var_pd / (pd_val * (1 - pd_val))
    rho_def = np.clip(rho_def, 0.001, 0.999)  
    rho_asset = invert_default_to_asset_correlation(pd_val, rho_def)
    rho_ts[g] = {'rho_default': rho_def, 'rho_asset': rho_asset, 'var_pd': var_pd}

rho_moment = np.array([rho_ts[g]['rho_asset'] for g in grades])

print("Moment-Matching Asset Correlations (from time-series variance)")
print(f"{'Grade':<8s} {'Var[p(X)]':>12s} {'ρ_default':>12s} {'ρ_asset':>12s}")
for i, g in enumerate(grades):
    print(f"  {g:<6s} {rho_ts[g]['var_pd']:>12.6f} {rho_ts[g]['rho_default']:>12.4f} "
          f"{rho_moment[i]:>12.4f}")

Moment-Matching Asset Correlations (from time-series variance)
Grade       Var[p(X)]    ρ_default      ρ_asset
  A          0.000376       0.0108       0.0552
  B          0.001631       0.0205       0.0616
  C          0.002538       0.0205       0.0478
  D          0.004211       0.0258       0.0512
  E          0.005922       0.0290       0.0509
  F          0.009252       0.0398       0.0649
  G          0.013103       0.0544       0.0869


In [178]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=grades, y=rho_moment,
    mode='lines+markers',
    line=dict(color='#e74c3c', width=3), marker=dict(size=10)
))

fig.update_layout(
    title="Asset Correlation Estimates: Moment Matching (Time-Series)",
    xaxis_title="Grade",
    yaxis_title="Asset Correlation (ρ)",
    template="plotly_white", height=420,
    legend=dict(x=0.6, y=0.95)
)
fig.show()

In [179]:
# Choose final factor loadings 
# We use moment matching estimates; replace NaN with a conservative default (0.10)
rho_final = np.where(np.isnan(rho_moment), 0.10, rho_moment)
sqrt_rho = np.sqrt(rho_final)  # factor loadings

print("Final Factor Loadings (√ρ)")
for i, g in enumerate(grades):
    print(f" For  Grade {g}: ρ = {rho_final[i]:.4f}  →  √ρ = {sqrt_rho[i]:.4f}")

Final Factor Loadings (√ρ)
 For  Grade A: ρ = 0.0552  →  √ρ = 0.2349
 For  Grade B: ρ = 0.0616  →  √ρ = 0.2482
 For  Grade C: ρ = 0.0478  →  √ρ = 0.2185
 For  Grade D: ρ = 0.0512  →  √ρ = 0.2262
 For  Grade E: ρ = 0.0509  →  √ρ = 0.2256
 For  Grade F: ρ = 0.0649  →  √ρ = 0.2548
 For  Grade G: ρ = 0.0869  →  √ρ = 0.2948


---
## 4. Joint Default Probability Distribution

To write the full joint distribution we need: $PD_k$ (marginal default rates), $\rho_k$ (asset correlations) & $n_k$ (clusters size). 

### The joint distribution

$$\boxed{P(D_1 \leq d_1, \ldots, D_K \leq d_K) = \int_{-\infty}^{\infty} \prod_{k=1}^{K} \sum_{j=0}^{d_k} \binom{n_k}{j} p_k(x)^j (1-p_k(x))^{n_k - j}\; \phi(x)\, dx}$$

where the **conditional PD** is:
$$p_k(x) = \Phi\!\left(\frac{\Phi^{-1}(PD_k) - \sqrt{\rho_k}\, x}{\sqrt{1-\rho_k}}\right)$$

We:
1. Pick a state of the economy $X = x$ (drawing from N(0,1) in our case )
2. Compute the conditional default probability $p_k(x)$ for each cluster — a bad economy ($x \ll 0$) pushes all $PDs$ up
3. Given $x$, defaults are independent Binomials— easier to compute
4. Average over all possible economies by integrating against $\phi(x)$
5. The copula structure enters through the *shared* factor $X$ — it creates the tail dependence

In [180]:
# Conditional default probabilities as a function of X
def conditional_pd(x, pd_marginal, rho):
    c = ndtri(pd_marginal)          # default threshold
    numerator = c - np.sqrt(rho) * x
    denominator = np.sqrt(1 - rho)
    return stats.norm.cdf(numerator / denominator)

x_grid = np.linspace(-4, 4, 500)

fig = go.Figure()
for i, g in enumerate(grades):
    cpd = conditional_pd(x_grid, pds[i], rho_final[i])
    fig.add_trace(go.Scatter(
        x=x_grid, y=cpd, mode='lines', name=f'Grade {g} (PD={pds[i]:.1%})',
        line=dict(width=2.5)
    ))

fig.add_vline(x=0, line_dash="dash", line_color="gray",
              annotation_text="Normal (X=0)")
fig.add_vline(x=-2, line_dash="dot", line_color="red",
              annotation_text="Bad (X=−2)")
fig.add_vline(x=-3, line_dash="dot", line_color="darkred",
              annotation_text="Severe (X=−3)")
fig.update_layout(
    title="Conditional Default Probability p_k(x) vs. Systematic Factor X",
    xaxis_title="Systematic Factor X (← recession | expansion →)",
    yaxis_title="Conditional Default Probability",
    yaxis_tickformat=".0%",
    template="plotly_white", height=500,
    legend=dict(x=0.81, y=0.909)
)
fig.show()

In [181]:
print("Conditional Default Rates - Various Economic Scenarios")
x_scenarios = {'Good (+2σ)': 2, 'Ok (+1σ)': 1, 'Normal (0)': 0,
               'Recession (−1σ)': -1, 'Bad (−2σ)': -2, 'Severe (−3σ)': -3}

header = f"{'Scenario':<20s}" + "".join(f"{'Grade ' + g:>10s}" for g in grades)
print(header)
for name, x in x_scenarios.items():
    row = f"  {name:<18s}"
    for i in range(K):
        cpd = conditional_pd(x, pds[i], rho_final[i])
        row += f"{cpd:>10.2%}"
    print(row)

Conditional Default Rates - Various Economic Scenarios
Scenario               Grade A   Grade B   Grade C   Grade D   Grade E   Grade F   Grade G
  Good (+2σ)             0.98%     2.77%     6.25%     9.52%    14.80%    19.02%    19.23%
  Ok (+1σ)               1.82%     4.85%     9.51%    14.08%    20.80%    26.98%    28.75%
  Normal (0)             3.21%     8.02%    13.87%    19.92%    28.03%    36.31%    40.04%
  Recession (−1σ)        5.37%    12.56%    19.43%    27.01%    36.30%    46.55%    52.24%
  Bad (−2σ)              8.56%    18.64%    26.17%    35.19%    45.27%    57.02%    64.24%
  Severe (−3σ)          13.00%    26.26%    33.93%    44.12%    54.49%    67.01%    74.97%


Under a 3σ downturn, portfolio-wide default rates can be 5–10× the 'normal' level

---
## 5. Monte Carlo Simulation & Stress Testing

Now we simulate from the joint distribution using Monte Carlo. The algorithm goes like:

1. Draw $M$ samples of $X \sim N(0,1)$
2. For each draw $X^{(m)}$, compute $p_k(X^{(m)})$ for all clusters
3. Draw $D_k^{(m)} \sim \text{Binomial}(n_k, p_k(X^{(m)}))$ — conditionally independent
4. Compute portfolio loss: $L^{(m)} = \sum_k D_k^{(m)} \cdot LGD_k \cdot EAD_k$

We can then stress test by conditioning on $X \leq x^*$ (i.e., only looking at recession scenarios).

In [189]:
M = 50_000  # number of scenarios

ead_k = grade_stats['avg_loan'].values   # EAD per loan by cluster
lgd = 0.60  # just assumed that LGD = 60%                              

scale_factor = 1000 / n_k.sum() # cluster sizes rescaled for tractability (proportional weights)
n_k_sim = np.maximum((n_k * scale_factor).astype(int), 10)

print(f"Simulation Parameters")
print(f"  Monte Carlo paths: {M:,}")
print(f"  LGD:               {lgd:.0%}")
print(f"  Simulated cluster sizes: {dict(zip(grades, n_k_sim))}")

X_draws = np.random.standard_normal(M)  # systematic factor draws

# For each scenario, we compute default counts and losses
default_counts = np.zeros((M, K), dtype=int)
losses = np.zeros(M)

for m in range(M):
    x = X_draws[m]
    for k in range(K):
        # Conditional PD given X
        cpd = conditional_pd(x, pds[k], rho_final[k])
        # Draw number of defaults (conditionally indep Binomial)
        default_counts[m, k] = np.random.binomial(n_k_sim[k], cpd)
        # Loss contribution
        losses[m] += default_counts[m, k] * lgd * ead_k[k]

# Convert to portfolio-level default rate
total_obligors = n_k_sim.sum()
portfolio_default_rate = default_counts.sum(axis=1) / total_obligors

print(f"  Portfolio default rate: mean = {portfolio_default_rate.mean():.2%}, "
      f"std = {portfolio_default_rate.std():.2%}")
print(f"  Loss distribution:     mean = ${losses.mean():,.0f}, "
      f"std = ${losses.std():,.0f}")

Simulation Parameters
  Monte Carlo paths: 50,000
  LGD:               60%
  Simulated cluster sizes: {'A': np.int64(192), 'B': np.int64(293), 'C': np.int64(287), 'D': np.int64(143), 'E': np.int64(59), 'F': np.int64(18), 'G': np.int64(10)}
  Portfolio default rate: mean = 13.03%, std = 4.75%
  Loss distribution:     mean = $1,227,342, std = $440,545


In [183]:
var_95 = np.percentile(losses, 95)
var_99 = np.percentile(losses, 99)
var_999 = np.percentile(losses, 99.9)
cvar_99 = losses[losses >= var_99].mean()
el = losses.mean()

print("Portfolio Risk Measures")
print(f"  Expected Loss (EL):    ${el:>12,.0f}")
print(f"  VaR 95%:               ${var_95:>12,.0f}")
print(f"  VaR 99%:               ${var_99:>12,.0f}")
print(f"  VaR 99.9%:             ${var_999:>12,.0f}")
print(f"  CVaR/ES 99%:           ${cvar_99:>12,.0f}")
print(f"  Economic Capital:      ${var_99 - el:>12,.0f}  (VaR99 − EL)")

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=losses, nbinsx=150, name='Loss Distribution',
    marker_color='rgba(52, 152, 219, 0.6)',
    histnorm='probability density'
))

for val, name, color, dash in [
    (el,      'Expected Loss',  '#2ecc71', 'dash'),
    (var_95,  'VaR 95%',        '#f39c12', 'dash'),
    (var_99,  'VaR 99%',        '#e74c3c', 'solid'),
    (var_999, 'VaR 99.9%',      '#8e44ad', 'dot'),
    (cvar_99, 'CVaR 99%',       '#e74c3c', 'dashdot'),
]:
    fig.add_vline(x=val, line_dash=dash, line_color=color,
                  annotation_text=name, annotation_position="top right")

fig.update_layout(
    title="Simulated Portfolio Loss Distribution",
    xaxis_title="Portfolio Loss ($)",
    yaxis_title="Density",
    template="plotly_white", height=500,
    showlegend=False
)
fig.show()

Portfolio Risk Measures
  Expected Loss (EL):    $   1,225,446
  VaR 95%:               $   2,025,631
  VaR 99%:               $   2,473,246
  VaR 99.9%:             $   2,977,868
  CVaR/ES 99%:           $   2,693,320
  Economic Capital:      $   1,247,799  (VaR99 − EL)


### Stress Testing the Latent Factor - Recession scenario


We condition on $X \leq x^*$ for various stress levels and examine the resulting loss distribution

In [184]:
stress_levels = {
    'Baseline (all X)':    (-np.inf, np.inf),
    'Mild stress (X ≤ 0)': (-np.inf, 0),
    'Moderate (X ≤ −1)':   (-np.inf, -1),
    'Severe (X ≤ −2)':     (-np.inf, -2),
    'Extreme (X ≤ −3)':    (-np.inf, -3),
}

fig = go.Figure()
stress_colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']

print("Stress Test Results")
print(f"{'Scenario':<25s} {'# Paths':>10s} {'Mean Loss':>12s} {'VaR 99%':>12s} "
      f"{'Default Rate':>14s}")

for (name, (x_lo, x_hi)), color in zip(stress_levels.items(), stress_colors):
    mask = (X_draws >= x_lo) & (X_draws <= x_hi)
    if mask.sum() < 10:
        continue
    stressed_losses = losses[mask]
    stressed_dr = portfolio_default_rate[mask]

    print(f"  {name:<23s} {mask.sum():>10,d} ${stressed_losses.mean():>11,.0f} "
          f"${np.percentile(stressed_losses, 99):>11,.0f} "
          f"{stressed_dr.mean():>14.2%}")

    fig.add_trace(go.Histogram(
        x=stressed_losses, name=name, opacity=0.5,
        marker_color=color, histnorm='probability density',
        nbinsx=80
    ))

print("-" * 80)

fig.update_layout(
    title="Portfolio Loss Distribution Under Stress Scenarios",
    xaxis_title="Portfolio Loss ($)",
    yaxis_title="Density",
    barmode='overlay',
    template="plotly_white", height=500,
    legend=dict(x=0.55, y=0.95)
)
fig.show()

Stress Test Results
Scenario                     # Paths    Mean Loss      VaR 99%   Default Rate
  Baseline (all X)            50,000 $  1,225,446 $  2,473,246         13.01%
  Mild stress (X ≤ 0)         24,888 $  1,565,514 $  2,640,003         16.67%
  Moderate (X ≤ −1)            7,858 $  1,952,554 $  2,901,881         20.87%
  Severe (X ≤ −2)              1,087 $  2,486,368 $  3,275,924         26.69%
  Extreme (X ≤ −3)                63 $  3,078,979 $  3,483,929         33.22%
--------------------------------------------------------------------------------


In [185]:
print("Default Rate by Cluster Under Stress Scenarios")
header = f"{'Scenario':<25s}" + "".join(f"{'Cluster ' + g:>10s}" for g in grades)
print(header)

for name, (x_lo, x_hi) in stress_levels.items():
    mask = (X_draws >= x_lo) & (X_draws <= x_hi)
    if mask.sum() < 10:
        continue
    row = f"  {name:<23s}"
    for k in range(K):
        dr = default_counts[mask, k].mean() / n_k_sim[k]
        row += f"{dr:>10.2%}"
    print(row)

Default Rate by Cluster Under Stress Scenarios
Scenario                  Cluster A Cluster B Cluster C Cluster D Cluster E Cluster F Cluster G
  Baseline (all X)            3.59%     8.66%    14.41%    20.48%    28.48%    36.60%    40.47%
  Mild stress (X ≤ 0)         5.05%    11.78%    18.38%    25.66%    34.66%    44.46%    49.77%
  Moderate (X ≤ −1)           7.03%    15.68%    22.86%    31.23%    40.99%    52.18%    58.65%
  Severe (X ≤ −2)            10.22%    21.47%    28.90%    38.63%    48.96%    61.28%    68.40%
  Extreme (X ≤ −3)           14.34%    28.58%    35.79%    45.72%    57.04%    68.43%    74.92%
